# Volume 8, Chapter 6: Homology (Python companion)

Homology computed by the Smith normal form, persistent homology of point clouds, and a discrete ham sandwich cut.

*Before each solution, predict what the output will look like.*

## Exercises

**P6.1 (★) Homology by the Smith normal form.** (a) Implement the Smith normal form of an integer matrix and, with Proposition 6.5, the integral homology of a finite simplicial complex given by its maximal simplices. (b) Compute the homology of the boundary of a tetrahedron, of the $3\times3$ torus of Figure 6.1, of a $6$-vertex projective plane and of a Klein bottle built on a $4\times4$ grid, and check the Euler characteristics. (c) Check numerically that torsion is invisible with coefficients in $\Q$ but visible with coefficients in $\Z/2$.

**P6.2 (★★) CS application: persistent homology.** A cloud of points sampled near a circle has no holes as a set, but it ``looks like'' a circle. Topological data analysis makes this precise: for each scale $r$, the Vietoris--Rips complex $R_r$ has a simplex for every set of points with pairwise distances at most $r$; as $r$ grows, holes are born and die, and a long-lived hole is a real feature. (a) Implement the persistence algorithm: order all simplices by the scale at which they appear, reduce the boundary matrix over $\Z/2$ column by column, and read off the pairs (birth, death). (b) Apply it to $40$ noisy points on a circle and on two disjoint circles, and plot the barcodes of $H_0$ and $H_1$. (c) Check that the number of bars alive at scale $r$ equals the Betti number of $R_r$ computed directly.

**P6.3 (★) CS application: a ham sandwich cut.** Implement the discrete ham sandwich cut of Exercise 6.6(b) by brute force: for two point sets in general position with an even number of points each, try the lines through one point of each set (and through two points of one set), and keep one that leaves at most half of each set strictly on each side. Test it on random sets, and count how many such lines exist.

## Solutions

### P6.1

In [ ]:
import itertools
from fractions import Fraction

def smith_diagonal(M):
    """invariant factors (nonzero diagonal of the Smith normal form) of an integer matrix, by row and column operations"""
    A = [row[:] for row in M]
    m, n = len(A), (len(A[0]) if A else 0)
    diag, t = [], 0
    while t < min(m, n):
        # find a nonzero entry of least absolute value in the remaining block
        piv = min(((abs(A[i][j]), i, j) for i in range(t, m) for j in range(t, n) if A[i][j]), default=None)
        if piv is None:
            break
        _, i, j = piv
        A[t], A[i] = A[i], A[t]
        for row in A:
            row[t], row[j] = row[j], row[t]
        done = False
        while not done:
            done = True
            for i in range(t + 1, m):                  # clear column t
                q = A[i][t] // A[t][t]
                if q:
                    A[i] = [a - q * b for a, b in zip(A[i], A[t])]
                if A[i][t]:
                    A[t], A[i] = A[i], A[t]; done = False
            for j in range(t + 1, n):                  # clear row t
                q = A[t][j] // A[t][t]
                if q:
                    for row in A:
                        row[j] -= q * row[t]
                if A[t][j]:
                    for row in A:
                        row[t], row[j] = row[j], row[t]
                    done = False
            if done:                                   # make the pivot divide the rest of the block
                for i in range(t + 1, m):
                    for j in range(t + 1, n):
                        if A[i][j] % A[t][t]:
                            A[t] = [a + b for a, b in zip(A[t], A[i])]; done = False; break
                    if not done:
                        break
        diag.append(abs(A[t][t])); t += 1
    return diag

def complex_from(maximal):
    S = set()
    for s in maximal:
        s = tuple(sorted(s))
        for k in range(1, len(s) + 1):
            S.update(itertools.combinations(s, k))
    by_dim = {}
    for s in S:
        by_dim.setdefault(len(s) - 1, []).append(s)
    return {d: sorted(v) for d, v in by_dim.items()}

def boundary(K, k):
    rows, cols = K.get(k - 1, []), K.get(k, [])
    idx = {s: i for i, s in enumerate(rows)}
    M = [[0] * len(cols) for _ in rows]
    for j, s in enumerate(cols):
        for i in range(len(s)):
            M[idx[s[:i] + s[i + 1:]]][j] = (-1) ** i
    return M

def homology(K):
    top = max(K)
    diags = {k: smith_diagonal(boundary(K, k)) if k >= 1 else [] for k in range(top + 2)}
    res = {}
    for k in range(top + 1):
        c = len(K[k]); rk = len(diags[k]); rk1 = len(diags[k + 1])
        torsion = [d for d in diags[k + 1] if d > 1]
        res[k] = (c - rk - rk1, torsion)
    return res

def show(name, K):
    H = homology(K)
    chi = sum((-1) ** k * len(K[k]) for k in K)
    txt = ", ".join(f"H{k} = Z^{b}" + "".join(f" + Z/{d}" for d in t) for k, (b, t) in H.items())
    print(f"{name:22s} counts {[len(K[k]) for k in sorted(K)]}, chi = {chi:2d}:  {txt}")

def grid(m, twist=False):
    v = lambda i, j: (((-i) % m if twist and (j // m) % 2 else i % m), j % m)
    lab = {}
    def L(p):
        return lab.setdefault(p, len(lab))
    T = []
    for i in range(m):
        for j in range(m):
            a, b, c, d = v(i, j), v(i + 1, j), v(i, j + 1), v(i + 1, j + 1)
            T += [(L(a), L(b), L(d)), (L(a), L(c), L(d))]
    return T

tetra = [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]
torus = grid(3)
rp2 = [(1, 2, 3), (1, 3, 4), (1, 4, 5), (1, 5, 6), (1, 6, 2), (2, 3, 5), (3, 4, 6), (4, 5, 2), (5, 6, 3), (6, 2, 4)]
klein = grid(4, twist=True)
for name, T in [("sphere (tetrahedron)", tetra), ("torus 3x3", torus), ("projective plane", rp2), ("Klein bottle 4x4", klein)]:
    K = complex_from(T)
    assert len(K[2]) == len(T), "two triangles with the same vertices"
    edges_per = {}
    for t in T:
        for e in itertools.combinations(sorted(t), 2):
            edges_per[e] = edges_per.get(e, 0) + 1
    assert all(c == 2 for c in edges_per.values()), "not a closed surface"
    show(name, K)

In [ ]:
def rank_mod(M, p):
    """rank over Z/p (p prime) or over Q (p = 0)"""
    A = [[Fraction(x) if p == 0 else x % p for x in row] for row in M]
    r, n = 0, (len(A[0]) if A else 0)
    for j in range(n):
        piv = next((i for i in range(r, len(A)) if A[i][j]), None)
        if piv is None:
            continue
        A[r], A[piv] = A[piv], A[r]
        inv = 1 / A[r][j] if p == 0 else pow(A[r][j], p - 2, p)
        A[r] = [x * inv if p == 0 else x * inv % p for x in A[r]]
        for i in range(len(A)):
            if i != r and A[i][j]:
                f = A[i][j]
                A[i] = [a - f * b if p == 0 else (a - f * b) % p for a, b in zip(A[i], A[r])]
        r += 1
    return r

for name, T in [("projective plane", rp2), ("Klein bottle 4x4", klein), ("torus 3x3", torus)]:
    K = complex_from(T)
    for p, lab in [(0, "Q"), (2, "Z/2")]:
        r = {k: rank_mod(boundary(K, k), p) if k >= 1 else 0 for k in range(4)}
        b = [len(K[k]) - r[k] - r[k + 1] for k in range(3)]
        print(f"{name:18s} Betti numbers over {lab:3s}: {b}")

(a)--(b) The output gives $H_*(S^2)=(\Z,0,\Z)$, $H_*(T^2)=(\Z,\Z^2,\Z)$, $H_*(\R P^2)=(\Z,\Z/2,0)$ and $H_*(K)=(\Z,\Z\oplus\Z/2,0)$, with Euler characteristics $2,0,1,0$, matching Example 6.16 and Exercise 6.1. The assertions check that the grid constructions are honest simplicial complexes (no two triangles with the same vertices, every edge on exactly two triangles); a $2\times2$ grid would fail the first test, which is why Figure 6.1 uses $3\times3$.

(c) Over $\Q$ the projective plane has Betti numbers $(1,0,0)$, as if it had no holes at all, and the Klein bottle $(1,1,0)$: the torsion $\Z/2$ disappears. Over $\Z/2$ both get an extra class in degrees $1$ and $2$: the projective plane becomes $(1,1,1)$, the Klein bottle $(1,2,1)$, which is the homology used in the proof of the Borsuk--Ulam theorem. The Euler characteristic is the same with every coefficient field, as Proposition 6.5 predicts, since it is the alternating count of simplices.

### P6.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rips_filtration(P, rmax):
    n = len(P)
    D = np.linalg.norm(P[:, None] - P[None, :], axis=2)
    simplices = [(0.0, 0, (i,)) for i in range(n)]
    for i, j in itertools.combinations(range(n), 2):
        if D[i, j] <= rmax:
            simplices.append((D[i, j], 1, (i, j)))
    for i, j, k in itertools.combinations(range(n), 3):
        r = max(D[i, j], D[i, k], D[j, k])
        if r <= rmax:
            simplices.append((r, 2, (i, j, k)))
    simplices.sort()
    return simplices

def persistence(simplices):
    """standard column reduction over Z/2; columns stored as Python ints (bit sets)"""
    index = {s: t for t, (_, _, s) in enumerate(simplices)}
    low_owner, cols, pairs = {}, [], []
    for t, (r, d, s) in enumerate(simplices):
        col = 0
        if d > 0:
            for i in range(len(s)):
                col |= 1 << index[s[:i] + s[i + 1:]]
        while col:
            low = col.bit_length() - 1
            if low in low_owner:
                col ^= cols[low_owner[low]]
            else:
                low_owner[low] = t
                b = simplices[low]
                pairs.append((b[1], b[0], r))       # (dimension, birth, death)
                break
        cols.append(col)
    killed = {low for low in low_owner} | set(low_owner.values())
    for t, (r, d, s) in enumerate(simplices):
        if t not in killed:
            pairs.append((d, r, np.inf))
    return pairs

rng = np.random.default_rng(1)
th = rng.uniform(0, 2 * np.pi, 40)
circle = np.c_[np.cos(th), np.sin(th)] + 0.08 * rng.standard_normal((40, 2))
th2 = rng.uniform(0, 2 * np.pi, 40)
two = np.r_[np.c_[np.cos(th2[:20]), np.sin(th2[:20])] * 0.8 - [1.2, 0],
            np.c_[np.cos(th2[20:]), np.sin(th2[20:])] * 0.8 + [1.2, 0]] + 0.06 * rng.standard_normal((40, 2))

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for row, (name, P) in enumerate([("noisy circle", circle), ("two circles", two)]):
    S = rips_filtration(P, 2.2)
    pairs = persistence(S)
    axes[row, 0].scatter(P[:, 0], P[:, 1], s=10); axes[row, 0].set_aspect("equal"); axes[row, 0].set_title(name)
    y = 0
    for dim, col in ((0, "C0"), (1, "C1")):
        bars = sorted([(b, d) for dd, b, d in pairs if dd == dim and d - b > 1e-9], key=lambda x: -(min(x[1], 2.2) - x[0]))
        for b, d in bars[:25]:
            axes[row, 1].plot([b, min(d, 2.2)], [y, y], color=col); y += 1
    axes[row, 1].set_title(f"{name}: H0 (blue) and H1 (orange) bars"); axes[row, 1].set_xlabel("scale r")
    long1 = sorted([d - b for dd, b, d in pairs if dd == 1], reverse=True)[:3]
    print(f"{name}: {len(S)} simplices; three longest H1 bars: {np.round(long1, 3)}")
plt.tight_layout(); plt.show()

# (c) compare with Betti numbers of R_r computed directly
def betti_at(P, r):
    S = [s for s in rips_filtration(P, r)]
    K = {}
    for _, d, s in S:
        K.setdefault(d, []).append(s)
    K = {d: sorted(v) for d, v in K.items()}
    rk = {k: rank_mod(boundary(K, k), 2) if k in K and k >= 1 else 0 for k in range(4)}
    return [len(K.get(k, [])) - rk[k] - rk[k + 1] for k in range(2)]

pairs = persistence(rips_filtration(circle, 1.2))
for r in (0.2, 0.5, 0.9):
    alive = [sum(1 for dd, b, d in pairs if dd == k and b <= r < d) for k in (0, 1)]
    print(f"r = {r}: bars alive (H0, H1) = {alive},  Betti numbers of R_r = {betti_at(circle, r)}")

(a) Each simplex either creates a homology class (its reduced column is zero: its boundary was already a boundary) or kills the youngest class that its boundary still represents (the lowest one in the reduced column, ``the elder rule''). Over $\Z/2$ columns are bit sets and the reduction is a sequence of XORs; this is the algorithm of Edelsbrunner, Letscher and Zomorodian (2002) as presented by Zomorodian and Carlsson (2005).

(b) For the noisy circle all $H_0$ bars die early except one (the cloud becomes connected), and there is one long $H_1$ bar, born when the points close up around the circle and dying only when the scale is comparable to the diameter (the Rips complex then fills the disc); the other $H_1$ bars are short-lived noise. Two circles give two long $H_0$ bars until the gap between them is bridged, and two long $H_1$ bars. The barcode is stable: moving the points by $\varepsilon$ moves the ends of the bars by at most $2\varepsilon$ in this normalization (Cohen-Steiner, Edelsbrunner and Harer, 2007), which is why long bars can be trusted.

(c) The printed bar counts agree with the Betti numbers of $R_r$ over $\Z/2$ computed from scratch: persistence tracks all the complexes of the filtration with one reduction.

### P6.3

In [ ]:
def side(p, a, b):
    return np.sign((b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0]))

def ham_cuts(P, Q):
    pts = np.r_[P, Q]
    found = []
    for i, j in itertools.combinations(range(len(pts)), 2):
        a, b = pts[i], pts[j]
        sP = np.array([side(p, a, b) for p in P]); sQ = np.array([side(q, a, b) for q in Q])
        if max((sP > 0).sum(), (sP < 0).sum()) <= len(P) / 2 and max((sQ > 0).sum(), (sQ < 0).sum()) <= len(Q) / 2:
            found.append((i, j))
    return found

for trial in range(5):
    P = rng.standard_normal((10, 2)); Q = rng.standard_normal((10, 2)) + [1.5, 0.5]
    cuts = ham_cuts(P, Q)
    i, j = cuts[0]
    kind = "one point of each set" if (i < 10) != (j < 10) else "two points of one set"
    print(f"trial {trial}: {len(cuts)} ham sandwich lines through two of the 20 points; first one through {kind}")

Every trial finds cuts, as Exercise 6.6 guarantees. Why it suffices to search lines through two of the points: a ham sandwich line can be rotated and translated, keeping the side counts valid, until it touches two points (a standard sweeping argument: move it parallel until it hits a point, then rotate about that point until it hits another, never letting a point cross from one side to the other without passing through the line). Brute force costs $O(n^3)$; Lo, Matou\v sek and Steiger find a cut in $O(n)$ time by a prune-and-search on the dual arrangement of lines. Ham sandwich cuts are used to build balanced partition trees for range searching, and to split fairly between two parties a cake made of two ingredients.